In [ ]:
import sys
sys.path.append("src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import quantum_algorithms as qa

# ============================================================
# GLOBAL PLOT STYLE
# ============================================================

plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "legend.fontsize": 11,
})

# ============================================================
# EXPERIMENT SETTINGS
# ============================================================

shots = 1024
n_values = range(1, 8)
theta_values = np.linspace(0, np.pi, 181)

axes = {
    "X": (1, 0, 0),
    "Y": (0, 1, 0),
    "Z": (0, 0, 1),
}

functions = {
    "constant_0": qa.deutsch_jozsa.f_constant_0,
    "constant_1": qa.deutsch_jozsa.f_constant_1,
    "balanced_parity": qa.deutsch_jozsa.f_balanced_parity,
}

error_positions = {
    "E1_before_H": qa.deutsch_jozsa.deutsch_jozsa_error1,
    "E2_after_first_H": qa.deutsch_jozsa.deutsch_jozsa_error2,
    "E3_after_oracle": qa.deutsch_jozsa.deutsch_jozsa_error3,
    "E4_after_final_H": qa.deutsch_jozsa.deutsch_jozsa_error4,
}

label_map = {
    "no_error": "No Error",
    "E1_before_H": r"$\mathcal{E}_1$ before H",
    "E2_after_first_H": r"$\mathcal{E}_2$ after first H",
    "E3_after_oracle": r"$\mathcal{E}_3$ after oracle",
    "E4_after_final_H": r"$\mathcal{E}_4$ after final H",
}

error_order = [
    "no_error",
    "E1_before_H",
    "E2_after_first_H",
    "E3_after_oracle",
    "E4_after_final_H",
]

error_only_order = [
    "E1_before_H",
    "E2_after_first_H",
    "E3_after_oracle",
    "E4_after_final_H",
]

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def get_P0_from_samples(samples, shots):
    """Probability of measuring the all-zero input register."""
    return samples[0] / shots


def get_success_probability(function_name, P0):
    """
    Deutsch-Jozsa success probability.

    Constant functions succeed when the input register is 0...0.
    Balanced functions succeed when the input register is not 0...0.
    """

    if function_name in ["constant_0", "constant_1"]:
        return P0

    if function_name == "balanced_parity":
        return 1 - P0

    raise ValueError(f"Unknown function name: {function_name}")


def standard_error(p, shots):
    """Binomial standard error."""
    return np.sqrt(p * (1 - p) / shots)


# ============================================================
# RUN SIMULATION
# ============================================================

results = []

for n in n_values:
    print(f"Running n = {n}")

    target_qubit = n // 2

    for function_name, f in functions.items():

        # No-error reference
        state_ref = qa.deutsch_jozsa.deutsch_jozsa(n, f)

        samples_ref = qa.deutsch_jozsa.sample_measurements_input(
            state_ref,
            n,
            shots
        )

        P0_ref = get_P0_from_samples(samples_ref, shots)
        success_ref = get_success_probability(function_name, P0_ref)

        for theta in theta_values:
            theta_deg = np.degrees(theta)

            for axis_name, axis in axes.items():

                # Store no-error baseline for every theta and axis.
                # This makes plotting against theta easier.
                results.append({
                    "n": n,
                    "theta_rad": theta,
                    "theta_deg": theta_deg,
                    "axis": axis_name,
                    "target_qubit": target_qubit,
                    "function": function_name,
                    "error_position": "no_error",
                    "P0": P0_ref,
                    "success_probability": success_ref,
                    "std_error": standard_error(success_ref, shots),
                    "shots": shots,
                })

                # Rotation-error simulations
                for error_name, error_function in error_positions.items():

                    state_error = error_function(
                        n,
                        f,
                        theta,
                        target_qubit,
                        axis
                    )

                    samples_error = qa.deutsch_jozsa.sample_measurements_input(
                        state_error,
                        n,
                        shots
                    )

                    P0_error = get_P0_from_samples(samples_error, shots)

                    success_error = get_success_probability(
                        function_name,
                        P0_error
                    )

                    results.append({
                        "n": n,
                        "theta_rad": theta,
                        "theta_deg": theta_deg,
                        "axis": axis_name,
                        "target_qubit": target_qubit,
                        "function": function_name,
                        "error_position": error_name,
                        "P0": P0_error,
                        "success_probability": success_error,
                        "std_error": standard_error(success_error, shots),
                        "shots": shots,
                    })

df = pd.DataFrame(results)

df.to_csv(
    "deutsch_jozsa_rotation_error_results.csv",
    index=False
)

print("Simulation complete.")
print(df.head())

# ============================================================
# AVERAGE SUCCESS OVER FUNCTIONS
# ============================================================

df_avg = (
    df.groupby(
        [
            "n",
            "theta_rad",
            "theta_deg",
            "axis",
            "target_qubit",
            "error_position",
        ],
        as_index=False
    )
    .agg(
        average_success=("success_probability", "mean"),
        average_std_error=("std_error", "mean"),
        shots=("shots", "first"),
    )
)

df_avg.to_csv(
    "deutsch_jozsa_average_success_results.csv",
    index=False
)

print("Average success data created.")
print(df_avg.head())

# ============================================================
# PLOT FUNCTIONS
# ============================================================

def plot_error_positions(df, n_plot, axis_plot, function_plot):
    """
    Plot success probability for one function and compare all error positions.
    """

    plot_df = df[
        (df["n"] == n_plot) &
        (df["axis"] == axis_plot) &
        (df["function"] == function_plot)
    ]

    if plot_df.empty:
        print("No data found.")
        return

    plt.figure(figsize=(9, 6))

    for error_pos in error_order:
        subset = plot_df[plot_df["error_position"] == error_pos]

        plt.plot(
            subset["theta_deg"],
            subset["success_probability"],
            linewidth=2,
            label=label_map[error_pos]
        )

    plt.xlabel(r"Rotation angle $\theta$ (degrees)")
    plt.ylabel("Success probability")

    plt.title(
        f"Deutsch-Jozsa success under rotation errors\n"
        f"function={function_plot}, n={n_plot}, axis={axis_plot}, shots={shots}"
    )

    plt.xlim(0, 180)
    plt.ylim(0, 1.05)
    plt.grid(True, alpha=0.3)
    plt.legend(title="Error position")
    plt.tight_layout()

    filename = f"error_positions_n{n_plot}_{axis_plot}_{function_plot}.pdf"
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved figure: {filename}")


def plot_average_success(df_avg, n_plot, axis_plot, target_plot):
    """
    Plot average success probability over constant_0, constant_1,
    and balanced_parity.
    """

    plot_df = df_avg[
        (df_avg["n"] == n_plot) &
        (df_avg["axis"] == axis_plot) &
        (df_avg["target_qubit"] == target_plot)
    ]

    if plot_df.empty:
        print("No data found.")
        return

    plt.figure(figsize=(10, 6))

    for error_pos in error_order:
        subset = plot_df[plot_df["error_position"] == error_pos]

        plt.plot(
            subset["theta_deg"],
            subset["average_success"],
            linewidth=2,
            label=label_map[error_pos]
        )

    plt.xlabel(r"Rotation angle $\theta$ (degrees)")
    plt.ylabel("Average success probability")

    plt.title(
        f"Average Deutsch-Jozsa success under rotation errors\n"
        f"n={n_plot}, axis={axis_plot}, target qubit={target_plot}, shots={shots}"
    )

    plt.xlim(0, 180)
    plt.ylim(0, 1.05)
    plt.grid(True, alpha=0.3)
    plt.legend(title="Error position")
    plt.tight_layout()

    filename = f"average_success_n{n_plot}_{axis_plot}_q{target_plot}.pdf"
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved figure: {filename}")


def plot_error_positions_subplots(df, n_plot, axis_plot, function_plot):
    """
    Plot each error position separately.
    This is easier to read when curves overlap.
    """

    plot_df = df[
        (df["n"] == n_plot) &
        (df["axis"] == axis_plot) &
        (df["function"] == function_plot)
    ]

    if plot_df.empty:
        print("No data found.")
        return

    no_error = plot_df[plot_df["error_position"] == "no_error"]

    fig, axes_obj = plt.subplots(
        2,
        2,
        figsize=(12, 8),
        sharex=True,
        sharey=True
    )

    axes_obj = axes_obj.flatten()

    for ax, error_pos in zip(axes_obj, error_only_order):
        subset = plot_df[plot_df["error_position"] == error_pos]

        ax.plot(
            no_error["theta_deg"],
            no_error["success_probability"],
            linestyle="--",
            linewidth=2,
            label="No Error"
        )

        ax.plot(
            subset["theta_deg"],
            subset["success_probability"],
            linewidth=2,
            label=label_map[error_pos]
        )

        ax.set_title(label_map[error_pos])
        ax.set_xlim(0, 180)
        ax.set_ylim(0, 1.05)
        ax.grid(True, alpha=0.3)
        ax.legend()

    fig.suptitle(
        f"Separate error-position plots\n"
        f"function={function_plot}, n={n_plot}, axis={axis_plot}, shots={shots}",
        fontsize=15
    )

    fig.supxlabel(r"Rotation angle $\theta$ (degrees)")
    fig.supylabel("Success probability")

    plt.tight_layout()

    filename = f"subplots_errors_n{n_plot}_{axis_plot}_{function_plot}.pdf"
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved figure: {filename}")


def plot_error_impact(df, n_plot, axis_plot, function_plot):
    """
    Plot the loss in success probability:
    loss = no-error success - error success.
    """

    plot_df = df[
        (df["n"] == n_plot) &
        (df["axis"] == axis_plot) &
        (df["function"] == function_plot)
    ]

    if plot_df.empty:
        print("No data found.")
        return

    no_error = (
        plot_df[plot_df["error_position"] == "no_error"]
        [["theta_deg", "success_probability"]]
        .rename(columns={"success_probability": "success_no_error"})
    )

    plt.figure(figsize=(10, 6))

    for error_pos in error_only_order:
        subset = (
            plot_df[plot_df["error_position"] == error_pos]
            [["theta_deg", "success_probability"]]
        )

        merged = pd.merge(
            subset,
            no_error,
            on="theta_deg"
        )

        merged["success_loss"] = (
            merged["success_no_error"] -
            merged["success_probability"]
        )

        plt.plot(
            merged["theta_deg"],
            merged["success_loss"],
            linewidth=2,
            label=label_map[error_pos]
        )

    plt.xlabel(r"Rotation angle $\theta$ (degrees)")
    plt.ylabel("Loss in success probability")

    plt.title(
        f"Error impact on Deutsch-Jozsa success\n"
        f"function={function_plot}, n={n_plot}, axis={axis_plot}, shots={shots}"
    )

    plt.xlim(0, 180)
    plt.ylim(-0.05, 1.05)
    plt.grid(True, alpha=0.3)
    plt.legend(title="Error position")
    plt.tight_layout()

    filename = f"error_impact_n{n_plot}_{axis_plot}_{function_plot}.pdf"
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved figure: {filename}")


def plot_axis_comparison(df_avg, n_plot, target_plot, error_position_plot):
    """
    Compare X, Y, and Z rotation axes for one error position.
    """

    plot_df = df_avg[
        (df_avg["n"] == n_plot) &
        (df_avg["target_qubit"] == target_plot) &
        (df_avg["error_position"] == error_position_plot)
    ]

    if plot_df.empty:
        print("No data found.")
        return

    plt.figure(figsize=(10, 6))

    for axis_name in ["X", "Y", "Z"]:
        subset = plot_df[plot_df["axis"] == axis_name]

        plt.plot(
            subset["theta_deg"],
            subset["average_success"],
            linewidth=2,
            label=f"{axis_name}-axis rotation"
        )

    plt.xlabel(r"Rotation angle $\theta$ (degrees)")
    plt.ylabel("Average success probability")

    plt.title(
        f"Comparison of rotation axes\n"
        f"n={n_plot}, target qubit={target_plot}, "
        f"error={label_map[error_position_plot]}"
    )

    plt.xlim(0, 180)
    plt.ylim(0, 1.05)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    filename = f"axis_comparison_n{n_plot}_{error_position_plot}_q{target_plot}.pdf"
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved figure: {filename}")


def plot_n_comparison(df_avg, axis_plot, error_position_plot):
    """
    Compare average success probability for different input sizes n.
    """

    plot_df = df_avg[
        (df_avg["axis"] == axis_plot) &
        (df_avg["error_position"] == error_position_plot)
    ]

    if plot_df.empty:
        print("No data found.")
        return

    plt.figure(figsize=(10, 6))

    for n in sorted(plot_df["n"].unique()):
        subset = plot_df[plot_df["n"] == n]

        plt.plot(
            subset["theta_deg"],
            subset["average_success"],
            linewidth=2,
            label=f"n={n}"
        )

    plt.xlabel(r"Rotation angle $\theta$ (degrees)")
    plt.ylabel("Average success probability")

    plt.title(
        f"Scaling with number of qubits\n"
        f"axis={axis_plot}, error={label_map[error_position_plot]}"
    )

    plt.xlim(0, 180)
    plt.ylim(0, 1.05)
    plt.grid(True, alpha=0.3)
    plt.legend(title="Input qubits")
    plt.tight_layout()

    filename = f"n_comparison_{axis_plot}_{error_position_plot}.pdf"
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved figure: {filename}")


# ============================================================
# PLOT CALLS
# ============================================================

plot_error_positions(
    df,
    n_plot=5,
    axis_plot="X",
    function_plot="constant_1"
)

plot_error_positions(
    df,
    n_plot=5,
    axis_plot="Y",
    function_plot="constant_1"
)

plot_error_positions(
    df,
    n_plot=5,
    axis_plot="X",
    function_plot="balanced_parity"
)

plot_average_success(
    df_avg,
    n_plot=1,
    axis_plot="X",
    target_plot=0
)

plot_average_success(
    df_avg,
    n_plot=5,
    axis_plot="X",
    target_plot=2
)

plot_error_positions_subplots(
    df,
    n_plot=5,
    axis_plot="Y",
    function_plot="constant_1"
)

plot_error_impact(
    df,
    n_plot=5,
    axis_plot="Y",
    function_plot="constant_1"
)

plot_axis_comparison(
    df_avg,
    n_plot=5,
    target_plot=2,
    error_position_plot="E1_before_H"
)

plot_axis_comparison(
    df_avg,
    n_plot=5,
    target_plot=2,
    error_position_plot="E2_after_first_H"
)

plot_axis_comparison(
    df_avg,
    n_plot=5,
    target_plot=2,
    error_position_plot="E3_after_oracle"
)

plot_axis_comparison(
    df_avg,
    n_plot=5,
    target_plot=2,
    error_position_plot="E4_after_final_H"
)

plot_n_comparison(
    df_avg,
    axis_plot="X",
    error_position_plot="E1_before_H"
)

plot_n_comparison(
    df_avg,
    axis_plot="X",
    error_position_plot="E4_after_final_H"
)